# AI Tutor Pipeline

This notebook walks through the full AI Tutor pipeline: embedding, KG loading, hybrid retrieval, chain-of-thought prompting, and evaluation.

In [ ]:
import sys
sys.path.append('..')

from src.rag.embed_documents import build_vector_index, save_index
from src.kg.kg_loader import load_kg
from src.tutor.tutor_engine import ask_tutor, clear_memory
from src.evaluation.evaluation_metrics import evaluate_response

## 1. Build Vector Index
Embed all documents in `data/documents/` into a FAISS index.

In [ ]:
index, documents, filenames = build_vector_index('../data/documents')
print(f'Loaded {len(documents)} documents')
print(f'FAISS index size: {index.ntotal}')
print(f'Files: {filenames}')

## 2. Load Knowledge Graph
Load triples into Neo4j (requires Neo4j running).

In [ ]:
load_kg('../data/knowledge_graph/knowledge_triples.csv')

## 3. Ask a Question
Use the full pipeline: hybrid retrieval + CoT + ensemble prompting.

In [ ]:
question = 'What is recursion?'
answer, debug = ask_tutor(question, index, documents, filenames, use_cot=True)
print('ANSWER:')
print(answer)
print('\n---\nDEBUG:')
print(f'KG context length: {len(debug["kg_context"])}')
print(f'Doc context length: {len(debug["doc_context"])}')
print(f'Entities found: {debug["entities"]}')

## 4. Multi-turn Conversation
The tutor maintains conversation memory across questions.

In [ ]:
q2 = 'Give me an example'
answer2, debug2 = ask_tutor(q2, index, documents, filenames, use_cot=True)
print('ANSWER:')
print(answer2)

## 5. Evaluate Response
Evaluate the answer for concept coverage and reasoning quality.

In [ ]:
eval_result = evaluate_response(
    answer=answer,
    reference='Recursion is a programming technique where a function calls itself.',
    concepts=['Recursion', 'Base Case', 'Function', 'Call Stack']
)
print('Evaluation:')
for key, val in eval_result.items():
    print(f'  {key}: {val}')

## 6. Without CoT (Comparison)
Compare responses with and without chain-of-thought.

In [ ]:
clear_memory()
answer_no_cot, _ = ask_tutor('What is a binary search tree?', index, documents, filenames, use_cot=False)
print('Without CoT:')
print(answer_no_cot[:500])

In [ ]:
clear_memory()
answer_cot, _ = ask_tutor('What is a binary search tree?', index, documents, filenames, use_cot=True)
print('With CoT:')
print(answer_cot[:500])

## 7. Save Index (for reuse)
Cache the FAISS index to disk so it doesn't need rebuilding.

In [ ]:
save_index(index, documents, filenames)
print('Index saved.')